# 03 · NUPSI: inspect an export before using its numbers

**Objective:** Inspect a GeoTIFF's bands and color interpretation, identify visualizations, and sample a bounded table only after verifying numeric NDVI semantics.

**Route:** Manual acquisition / local files · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** a metadata report, band previews, and a maximum 1000-row numeric sample if the export is scientific NDVI. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/nupsi.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Obtain and inspect a small export

Open NUPSI through the [DestinE catalogue](https://platform.destine.eu/), select a small Sentinel-2 AOI and export. Save the file locally and set `INPUT_NUPSI`. Keep the export description and timestamp. The original file was four-band `uint8`; that often means a rendered RGBA image. Do not infer NDVI from red/green/blue color channels.


In [ ]:
import rasterio
from rasterio.enums import Resampling
from examples.raster import numeric_ndvi

path = input_path("INPUT_NUPSI")
with rasterio.open(path) as src:
    info = {"count": src.count, "dtype": src.dtypes, "crs": str(src.crs), "dimensions": src.shape, "descriptions": src.descriptions, "colors": [c.name for c in src.colorinterp], "scales": src.scales, "offsets": src.offsets}
    preview = src.read(out_shape=(src.count, min(256, src.height), min(256, src.width)), resampling=Resampling.nearest, masked=True)
print(info)
fig, axes = plt.subplots(1, len(preview), figsize=(4 * len(preview), 3), squeeze=False)
for index, band in enumerate(preview):
    axes[0, index].imshow(band)
    axes[0, index].set_title(f"Band {index + 1}: {info['colors'][index]}")
plt.show()


## 3. Validate numeric NDVI

Set `NUPSI_NDVI_BAND` only after the export documentation identifies that band as numeric NDVI. A four-band visualization deliberately stops here. Request the scientific numeric export rather than decoding a color palette. Raster scale/offset metadata are applied to the selected band.


In [ ]:
band_index = int(required_env("NUPSI_NDVI_BAND"))
with rasterio.open(path) as src:
    if not 1 <= band_index <= src.count:
        raise ValueError("Band index is outside the file.")
    # A bounded window avoids expanding millions of raster pixels into a DataFrame.
    window = rasterio.windows.Window(0, 0, min(256, src.width), min(256, src.height))
    raw = src.read(band_index, window=window, masked=True).astype(float).filled(np.nan)
    result = numeric_ndvi(raw, src.colorinterp, src.scales[band_index - 1], src.offsets[band_index - 1])
    transform = src.window_transform(window)
    rows, columns = np.nonzero(np.isfinite(result))
    take = np.linspace(0, len(rows) - 1, min(1000, len(rows)), dtype=int)
    x, y = rasterio.transform.xy(transform, rows[take], columns[take])
    sample = pd.DataFrame({"x": x, "y": y, "ndvi": result[rows[take], columns[take]]})
    sample_crs = str(src.crs)
assert len(sample) <= 1000
sample.head()


## 4. Export a clearly labelled sample


In [ ]:
(ROOT / "outputs").mkdir(exist_ok=True)
sample.to_csv(ROOT / "outputs" / "nupsi-bounded-sample.csv", index=False)
record_provenance("nupsi-inspection.json", input_file=path.name, band=band_index, coordinate_crs=sample_crs, sampling="up to 1000 pixels within upper-left 256x256 window", units="dimensionless NDVI", sample_count=len(sample))


## Interpret and verify

The sample represents a bounded inspection window, not an unbiased study-area mean. A valid numeric range alone cannot prove NDVI semantics: use the provider's description. Do not call a visualization an observation dataset.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [NUPSI official documentation](https://platform.destine.eu/docs/nupsi/doc/index.html)
- [Rasterio color interpretation](https://rasterio.readthedocs.io/en/stable/topics/color.html)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
